# Financial tweet sentiment

Given an English financial post, predict **bearish** (negative outlook), **bullish** (positive outlook), or **neutral**. This is supervised machine learning because the training examples come with labels. The target is the sentiment expressed in the text.

How well can Naive Bayes label financial posts? First, compare single words with single words plus pairs. Then test the chosen version on separate posts. Use the rule that always guesses neutral as a reference for the test score.

The notebook downloads and caches the pinned [Twitter Financial News Sentiment dataset](https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment) in `data/`.

## 0. Set up the notebook

In [1]:
import csv
from pathlib import Path
from os import getpid
from urllib.request import urlopen
import re
from collections import Counter

**Prepare the features and split.** `train_test_split` creates the development split. `CountVectorizer` turns text into term counts. `make_pipeline` ties that conversion to a classifier so both are fitted together on the training data.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

**Set up the classifier and baseline.** `MultinomialNB` learns from word counts. `DummyClassifier` predicts the most common training label.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB

**Evaluate predictions.** `accuracy_score` measures how often the prediction is correct. `f1_score` gives us the main comparison metric. `classification_report` gives a score for each label.

In [4]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

### Settings

In [5]:
# Use this page to find the dataset files.
DATA_URL = "https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment"
# Keep the same dataset version for each run.
REVISION = "ccbe24de388e287beb92dd393a335c376b350ac3"
# Store the downloaded CSV files in this folder.
DATA_DIR = Path("data")

In [6]:
# Map each numeric label to its meaning.
LABELS = {0: "bearish", 1: "bullish", 2: "neutral"}
# Use the same data split on each run.
SEED = 42

In [7]:
# Test single words and single words plus adjacent pairs.
feature_options = {
    1: "single words",
    2: "words and adjacent word pairs",
}

### Functions

In [8]:
# Get the local CSV or download and save it.
def dataset_file(filename):
    path = DATA_DIR / filename
    if not path.exists():
        url = f"{DATA_URL}/resolve/{REVISION}/{filename}"
        with urlopen(url, timeout=20) as response:
            content = response.read()
        DATA_DIR.mkdir(exist_ok=True)
        partial = path.with_suffix(f".{getpid()}.part")
        partial.write_bytes(content)
        partial.replace(path)
    return path

In [9]:
# Remove URLs and make the case and spaces consistent.
def clean(text):
    text = re.sub(r"https?://\S+", " ", text, flags=re.IGNORECASE)
    return " ".join(text.casefold().split())

In [10]:
# Read the CSV rows using the column names.
def read_csv_rows(path):
    with path.open(encoding="utf-8-sig", newline="") as source:
        return list(csv.DictReader(source))

In [11]:
# Clean the text, convert labels to integers, and skip empty posts.
def clean_post_rows(rows):
    posts = []
    for row in rows:
        text = clean(row["text"])
        if text:
            posts.append((text, int(row["label"])))
    return posts

In [12]:
# Group all labels for each cleaned post.
def group_labels_by_text(posts):
    grouped = {}
    for text, label in posts:
        grouped.setdefault(text, set()).add(label)
    return grouped

In [13]:
# Keep one copy of each post if all its labels agree.
def keep_consistent_posts(grouped):
    posts = {}
    for text, labels in grouped.items():
        if len(labels) == 1:
            posts[text] = next(iter(labels))
    return posts

In [14]:
# Load, clean, and filter posts from one dataset file.
def load_posts(filename):
    path = dataset_file(filename)
    rows = read_csv_rows(path)
    posts = clean_post_rows(rows)
    grouped = group_labels_by_text(posts)
    return keep_consistent_posts(grouped)

In [15]:
# Connect the word count step to the classifier.
def make_model(max_words, classifier):
    return make_pipeline(
        CountVectorizer(ngram_range=(1, max_words)),
        classifier,
    )

In [16]:
# Train the model and return its macro F1 on the development set.
def development_score(model):
    model.fit(train_text, train_labels)
    predictions = model.predict(dev_text)
    return f1_score(dev_labels, predictions, average="macro")

## 1. Load and clean the data

### Load the training pool

Make two aligned lists. The text at each position is the input, and the label at the same position is the answer the model learns from.

In [17]:
training_posts = load_posts("sent_train.csv")
texts = list(training_posts)
labels = list(training_posts.values())
print(f"Cleaned training pool: {len(texts):,} posts")

Cleaned training pool: 9,431 posts


### Check for class imbalance

Neutral makes up most of the training data. This motivates a most-frequent-label baseline and a metric that gives all classes equal weight.

In [18]:
counts = Counter(labels)
for label, name in LABELS.items():
    print(f"{name:7} {counts[label]:,} ({counts[label] / len(labels):.1%})")

bearish 1,428 (15.1%)
bullish 1,907 (20.2%)
neutral 6,096 (64.6%)


## 2. Choose settings on a development split

### Create a development set

Fit each version of Naive Bayes on 80% of the training pool and use the other 20% to compare settings. Stratification keeps the proportions of the three labels similar in both parts.

In [19]:
train_text, dev_text, train_labels, dev_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels,
)
print(f"{len(train_text):,} training posts and {len(dev_text):,} development posts")

7,544 training posts and 1,887 development posts


### Fit the reference model

The baseline learns only the most frequent training label. Here it always predicts neutral, so it will miss every bearish and bullish example.

In [20]:
baseline = DummyClassifier(strategy="most_frequent")
print(f"Most frequent label: {development_score(baseline):.3f}")

Most frequent label: 0.262


### Compare word counts

Train one version on single words and one with adjacent word pairs added. `alpha=1` adds one to each count so every term has a small chance under each label.

In [21]:
nb_scores = {}
for max_words, feature_name in feature_options.items():
    model = make_model(max_words, MultinomialNB(alpha=1))
    nb_scores[max_words] = development_score(model)
    print(f"Naive Bayes with {feature_name}: {nb_scores[max_words]:.3f}")

Naive Bayes with single words: 0.673


Naive Bayes with words and adjacent word pairs: 0.677


### Use the better feature setting

Select whichever setting has higher development macro F1 and use it to train the final Naive Bayes model.

In [22]:
selected_features = max(nb_scores, key=nb_scores.get)
print(f"Selected features: {feature_options[selected_features]}")

Selected features: words and adjacent word pairs


## 3. Refit the models and prepare the benchmark

### Gather the configurations

Use the most-frequent baseline and Naive Bayes with the selected word counts.

In [23]:
models = {
    "Baseline": baseline,
    "Naive Bayes": make_model(selected_features, MultinomialNB(alpha=1)),
}

### Refit on all cleaned training posts

The development split has served its purpose. Refit each model and its vocabulary on the full training pool, including the examples that were in development.

In [24]:
for model in models.values():
    model.fit(texts, labels)

print(f"Refitted {len(models)} models using {len(texts):,} posts")

Refitted 2 models using 9,431 posts


### Load and clean validation data

In [25]:
holdout = load_posts("sent_valid.csv")
print(f"Cleaned benchmark pool: {len(holdout):,} posts")

Cleaned benchmark pool: 2,377 posts


### Exclude benchmark posts the models have already seen

Check cleaned text against the training pool. Remove any exact overlaps so a duplicate cannot count as an unseen prediction.

In [26]:
test_text = [text for text in holdout if text not in training_posts]
test_labels = [holdout[text] for text in test_text]
print(f"Removed {len(holdout) - len(test_text)} benchmark texts found in training")
print(f"Benchmark: {len(test_text):,} posts")

Removed 32 benchmark texts found in training
Benchmark: 2,345 posts


## 4. Compare on the same benchmark posts

### Generate predictions

Pass the same benchmark texts to each fitted model. Each returns one predicted label per text, in the same order as the true labels.

In [27]:
predictions = {name: model.predict(test_text) for name, model in models.items()}

### Compare accuracy and macro F1

The always-neutral baseline shows why accuracy alone can be misleading. The clearer signal here is macro F1 because correct calls for rarer labels matter too.

In [28]:
print(f"{'Model':22} {'Accuracy':>8} {'Macro F1':>8}")
for name, predicted in predictions.items():
    accuracy = accuracy_score(test_labels, predicted)
    macro_f1 = f1_score(test_labels, predicted, average="macro")
    print(f"{name:22} {accuracy:8.1%} {macro_f1:8.3f}")

Model                  Accuracy Macro F1
Baseline                  65.4%    0.264
Naive Bayes               80.9%    0.706


### Report precision and recall by label

Check whether a model catches bearish examples and how trustworthy its bearish calls are. The support column shows how many benchmark examples actually have each label.

In [29]:
print(classification_report(
    test_labels, predictions["Naive Bayes"], labels=list(LABELS),
    target_names=list(LABELS.values()), digits=3, zero_division=0,
))

              precision    recall  f1-score   support

     bearish      0.833     0.429     0.566       338
     bullish      0.765     0.598     0.671       473
     neutral      0.816     0.958     0.881      1534

    accuracy                          0.809      2345
   macro avg      0.805     0.662     0.706      2345
weighted avg      0.808     0.809     0.793      2345

